### 1) 설비 트러블 문제 정의

[설비 트러블 문제 정의]
1) 프레스 슬라이드(위치 x, 속도 v)가 유압 압력 P에 의해 구동됨.
2) 정상 상태에서는: (m * a) ≈ (A*P - c*v - friction(x,v,P)) 관계를 만족
3) 트러블(고장) 시나리오:
  (A) 가이드/슬라이드 마찰 증가(윤활 불량, 마모) → friction 항 급증
  (B) 유압 누설/밸브 응답 저하 → P 동역학이 정상과 달라짐(압력 형성 지연/감소)
- 목표: "물리 위반(physics residual)" 관점에서 조기 이상탐지 + 항별 분해로 원인분류 + Jacobian으로 변수상관성 분석

### 2) 개선 목표 정의
1) 정상 데이터로 시스템 동역학을 학습하되, 뉴턴/유압 물리 제약을 반드시 만족하도록(Physics-Informed)
2) 추론 시:
   - Physics Residual(물리 잔차) 기반 이상 점수로 조기 이상탐지
   - 항별 분해(|A*P|, |c*v|, |friction_hat|)로 원인 후보를 정량화(원인분류)
   - Jacobian(가속도 방정식의 민감도)로 변수간 상관성/인과 단서를 제공
3) 결과를 시각화하여 엔지니어 관점의 해석(“마찰 항 붕괴”, “압력 동역학 붕괴”)이 가능하도록

In [1]:
# ============================================================
# Hybrid Physics-Informed Neural ODE (PIN-ODE)
# ============================================================
# 이 스크립트는 프레스 설비의 슬라이드 운동과 유압 압력 동역학을
# "물리 방정식 + 신경망"을 결합한 Neural ODE로 모델링한다.
#
# 주요 산출물:
# 1) 상태 예측 (위치 x, 속도 v, 압력 P)
# 2) Physics Residual 기반 이상탐지 점수
# 3) 항별 분해를 통한 고장 원인 후보 추정
# 4) Jacobian 기반 변수 영향도(상관성) 해석
#
# 즉, "이상인가?" 뿐 아니라
# "왜 이상인가?"까지 설명 가능한 모델을 만드는 것이 목적이다.
# ============================================================

# ------------------------------------------------------------
# 1) 기본 라이브러리 import
# ------------------------------------------------------------
import numpy as np
import torch

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

np.random.seed(42)
# numpy 난수 고정

torch.manual_seed(42)
# torch 난수 고정

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device : ', device)
# GPU 사용 가능 시 cuda, 아니면 cpu

device :  cpu


In [6]:
# ------------------------------------------------------------
# 2. 물리 상수 및 시뮬레이션 기본 설정
# ------------------------------------------------------------
m = 50.0
# 슬라이드 + 금형 + 부하를 하나의 질량으로 본 등가 질량 값이다.
# 뉴턴 법칙 m * a = F 에서 사용된다.

A = 0.8
# 유압 면적 계수로, 압력 P가 힘으로 변환될 때 F = A * P 관계를 가진다.

c = 12.0
# 선형 감쇠 계수로, 속도에 비례하는 저항력 c * v 를 나타낸다.

dt = 0.02
# 샘플링 시간 간격으로, 0.02초는 50Hz 센서를 의미한다.

T = 30.0
# 하나의 시뮬레이션(프레스 사이클 포함)의 전체 길이(초)이다.

N = int(T / dt)
# 전체 시계열 데이터 포인트 개수를 계산한다.

t = np.arange(N) * dt
# 각 데이터 포인트에 대응하는 시간축을 생성한다.

In [7]:
# ------------------------------------------------------------
# 3. 제어 입력 u(t) 정의 (밸브/서보 명령)
# ------------------------------------------------------------
u = np.zeros(N)
# u(t)는 밸브 개도 또는 서보 명령을 의미하며, 0~1 범위로 가정한다.

def add_pulse(start_s, end_s, amp):
    # 특정 시간 구간(start_s~end_s)에
    # 밸브 입력을 amp 값으로 주기 위한 유틸리티 함수이다.
    s = int(start_s / dt)
    # 시작 시간을 인덱스로 변환한다.
    e = int(end_s / dt)
    # 종료 시간을 인덱스로 변환한다.
    u[s:e] = amp
    # 해당 구간의 입력 값을 amp로 설정한다.

add_pulse(1.0, 6.0, 0.8)
# 첫 번째 프레스 스트로크 구간을 모사한다.

add_pulse(10.0, 15.0, 0.9)
# 두 번째 프레스 스트로크 구간이다.

add_pulse(19.0, 24.0, 0.85)
# 세 번째 프레스 스트로크 구간이다.

In [8]:
# ------------------------------------------------------------
# 4. 데이터 생성을 위한 실제(정답) 마찰 모델
# ------------------------------------------------------------
def true_friction(x, v, mode="normal"):
    # 실제 설비에는 존재하지만 우리가 정확히 모른다고 가정하는
    # 비선형 마찰/접촉 손실 모델이다.

    coulomb = 25.0 * np.tanh(2.5 * v)
    # 속도 부호에 따라 포화되는 쿨롬 마찰 성분을 tanh로 부드럽게 근사한다.

    viscous = 8.0 * v
    # 속도에 비례하는 점성 마찰 성분이다.

    contact = 18.0 * np.tanh(3.0 * (x - 0.15))
    # 슬라이드가 특정 위치를 지나면
    # 접촉이나 하중 증가가 발생하는 효과를 모델링한다.

    base = coulomb + viscous + contact
    # 정상 상태의 기본 마찰력이다.

    if mode == "normal":
        # 정상 상태라면 기본 마찰을 그대로 사용한다.
        return base
    else:
        # 고장 상태에서는 마찰이 전반적으로 증가한다고 가정한다.
        return 1.8 * base + 15.0 * np.tanh(3.0 * v)


In [9]:
# ------------------------------------------------------------
# 5. 데이터 생성을 위한 실제(정답) 압력 동역학
# ------------------------------------------------------------
def true_pressure(P, u, mode="normal"):
    # 유압 시스템의 압력 변화율 dP/dt 를 단순화하여 모델링한다.

    if mode == "normal":
        ku, kleak = 220.0, 3.5
        # 정상 상태: 압력 형성 능력은 크고 누설은 작다.
    else:
        ku, kleak = 160.0, 6.0
        # 고장 상태: 밸브 응답 저하 또는 누설 증가를 의미한다.

    sat = 0.002 * max(P - 160.0, 0.0)**2
    # 압력이 너무 높아질 경우
    # 릴리프 밸브나 포화 효과로 더 빠르게 압력이 빠지도록 만든 항이다.

    return ku * u - kleak * P - sat
    # 최종 압력 변화율을 반환한다.

In [10]:
# ------------------------------------------------------------
# 6. 시계열 데이터 생성 함수
# ------------------------------------------------------------
def simulate(fault=None, fs=0.0, fe=0.0):
    # 하나의 프레스 사이클 시계열을 생성한다.
    # fault가 지정되면 특정 구간(fs~fe)에 고장을 주입한다.

    x = np.zeros(N)
    # 슬라이드 위치 배열

    v = np.zeros(N)
    # 슬라이드 속도 배열

    P = np.zeros(N)
    # 유압 압력 배열

    for k in range(N - 1):
        # 시간 축을 따라 상태를 순차적으로 계산한다.

        in_fault = fault is not None and fs <= k * dt <= fe
        # 현재 시점이 고장 구간에 속하는지 판단한다.

        fr = true_friction(
            x[k], v[k],
            mode="fault" if (fault == "friction" and in_fault) else "normal"
        )
        # 마찰 고장일 경우에만 마찰 모델을 고장 모드로 바꾼다.

        a = (A * P[k] - c * v[k] - fr) / m
        # 뉴턴 법칙을 이용해 가속도를 계산한다.

        dP = true_pressure(
            P[k], u[k],
            mode="fault" if (fault == "pressure" and in_fault) else "normal"
        )
        # 압력 고장일 경우에만 압력 모델을 고장 모드로 바꾼다.

        v[k+1] = v[k] + dt * a
        # 속도를 적분하여 다음 시점의 속도를 계산한다.

        x[k+1] = x[k] + dt * v[k]
        # 위치를 적분하여 다음 시점의 위치를 계산한다.

        P[k+1] = max(P[k] + dt * dP, 0.0)
        # 압력을 적분하되 음수로 내려가지 않도록 한다.

    return np.stack([x, v, P], axis=1)
    # (N,3) 형태의 시계열 데이터를 반환한다.

In [11]:
# ------------------------------------------------------------
# 7. 학습용/테스트용 데이터 생성
# ------------------------------------------------------------
train_trajs = [simulate() for _ in range(16)]
# 정상 상태 데이터만 여러 회차 생성하여 학습에 사용한다.

test_fric = simulate("friction", 12.0, 18.0)
# 마찰 고장 테스트 데이터

test_pres = simulate("pressure", 12.0, 18.0)
# 압력 고장 테스트 데이터

In [18]:
test_fric

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  3.03767284e-03,  0.00000000e+00],
       [ 6.07534568e-05,  5.97510393e-03,  0.00000000e+00],
       ...,
       [ 1.39782152e+00, -2.36079216e-01,  2.32921338e-08],
       [ 1.39309994e+00, -2.36082137e-01,  2.16616845e-08],
       [ 1.38837830e+00, -2.36084751e-01,  2.01453666e-08]],
      shape=(1500, 3))

In [19]:
# ------------------------------------------------------------
# 8. Hybrid PIN-ODE 모델 정의
# ------------------------------------------------------------
def mlp(in_dim):
    # 다층 퍼셉트론(MLP)을 간단히 생성하는 함수이다.
    return nn.Sequential(
        nn.Linear(in_dim, 64),
        nn.Tanh(),
        nn.Linear(64, 64),
        nn.Tanh(),
        nn.Linear(64, 1)
    )

class HybridPINODE(nn.Module):
    # 물리 방정식과 신경망을 결합한 Neural ODE 모델이다.

    def __init__(self):
        super().__init__()

        self.f_theta = mlp(3)
        # 위치, 속도, 압력을 입력으로 받아
        # 미지의 마찰 항을 추정하는 신경망이다.

        self.dp_corr = mlp(4)
        # 압력 물리 모델의 오차를 보정하기 위한 신경망이다.

    def dynamics(self, x, u):
        # 상태 미분 dx/dt = f(x,u)를 계산하는 함수이다.

        pos, vel, P = x[:,0:1], x[:,1:2], x[:,2:3]
        # 상태 벡터에서 위치, 속도, 압력을 분리한다.

        fr = self.f_theta(torch.cat([pos, vel, P], 1))
        # 마찰 항을 신경망으로 추정한다.

        dP_phys = 220.0 * u - 3.5 * P
        # 정상 상태 기준의 압력 물리 모델이다.

        dP = dP_phys + self.dp_corr(torch.cat([pos, vel, P, u], 1))
        # 물리 모델 + NN 보정항으로 압력 동역학을 구성한다.

        dx = vel
        # 물리적으로 위치의 시간 미분은 속도이다.

        dv = (A * P - c * vel - fr) / m
        # 물리적으로 가속도는 힘의 합을 질량으로 나눈 값이다.

        return torch.cat([dx, dv, dP], 1), fr
        # 상태 변화율과 마찰 추정값을 함께 반환한다.

In [20]:
# ------------------------------------------------------------
# 9. RK4 적분기
# ------------------------------------------------------------
def rk4(model, x, u):
    # Neural ODE를 수치적으로 적분하기 위한 4차 룽게-쿠타 방법이다.

    k1, _ = model.dynamics(x, u)
    k2, _ = model.dynamics(x + 0.5 * dt * k1, u)
    k3, _ = model.dynamics(x + 0.5 * dt * k2, u)
    k4, _ = model.dynamics(x + dt * k3, u)

    return x + dt / 6.0 * (k1 + 2*k2 + 2*k3 + k4)
    # 네 개의 기울기를 가중 평균하여 다음 상태를 계산한다.

In [28]:
# ------------------------------------------------------------
# 10. 모델 학습
# ------------------------------------------------------------
model = HybridPINODE().to(device)
# 모델을 생성하고 연산 디바이스로 이동시킨다.

opt = optim.Adam(model.parameters(), lr=2e-3)
# Adam 옵티마이저를 사용하여 파라미터를 학습한다.

for ep in range(10):
    # 전체 학습 epoch 200번 반복

    total_loss = 0.0
    # epoch 동안의 손실을 누적한다.

    for seq in train_trajs:
        # 각 정상 시계열에 대해 학습을 수행한다.

        X = torch.tensor(seq, dtype=torch.float32, device=device)
        # numpy 데이터를 torch 텐서로 변환한다.

        U = torch.tensor(u[:,None], dtype=torch.float32, device=device)
        # 입력 u도 torch 텐서로 변환한다.

        xh = X[0:1]
        # 초기 상태를 시작점으로 설정한다.

        preds = []
        # 예측 시계열을 저장할 리스트이다.

        for k in range(N):
            preds.append(xh)
            xh = rk4(model, xh, U[k:k+1])
            # ODE 적분을 통해 다음 상태를 예측한다.

        preds = torch.cat(preds)
        # 예측 결과를 하나의 텐서로 결합한다.

        data_loss = ((preds - X)**2).mean()
        # 관측 데이터와 예측 데이터의 차이를 MSE로 계산한다.

        # -----------------------------
        # (1) t 시점 속도 v(t)
        # -----------------------------
        v_t = X[:-1, 1]
        # 길이: N-1
        
        # -----------------------------
        # (2) t+1 시점 속도 v(t+1)
        # -----------------------------
        v_tp1 = X[1:, 1]
        # 길이: N-1
        
        # -----------------------------
        # (3) 가속도 a(t) ≈ (v(t+1)-v(t))/dt
        # -----------------------------
        acc = (v_tp1 - v_t) / dt
        # 길이: N-1
        
        # -----------------------------
        # (4) t 시점 상태로 마찰 추정
        # -----------------------------
        _, fr_hat = model.dynamics(X[:-1], U[:-1])
        fr_hat = fr_hat.squeeze()
        # 길이: N-1
        
        # -----------------------------
        # (5) t 시점 물리 RHS
        # -----------------------------
        rhs = A * X[:-1, 2] - c * v_t - fr_hat
        # 길이: N-1
        
        # -----------------------------
        # (6) Physics Residual (차원 일치!)
        # -----------------------------
        phys_res = m * acc - rhs

        # 물리 방정식의 좌변과 우변 차이를 Physics Residual로 계산한다.

        phys_loss = (phys_res**2).mean()
        # Physics Residual의 제곱 평균을 물리 손실로 사용한다.

        loss = data_loss + 3.0 * phys_loss
        # 데이터 적합과 물리 제약을 동시에 만족하도록 손실을 구성한다.

        opt.zero_grad()
        loss.backward()
        opt.step()
        # 역전파 및 파라미터 업데이트를 수행한다.

        total_loss += loss.item()

    if ep % 50 == 0:
        print(f"Epoch {ep:03d} | Loss {total_loss:.4f}")
        # 학습 진행 상황을 출력한다.

Epoch 000 | Loss 30425.0264


### 결과 해석 (현업 관점 요약)

-정상 상태
예측과 관측이 거의 일치
Physics Residual이 전 구간에서 낮음

-마찰 고장
압력은 정상이나 가속도가 나오지 않음
friction̂ 항 증가 + residual 급증
--> 가이드 마모 / 윤활 불량 의심

-압력 고장
마찰은 정상이나 압력 형성이 부족
압력 예측 오차 + residual 증가
--> 밸브 지연 / 누설 의심

In [29]:
# ------------------------------------------------------------
# 11. 학습 성능 시각화
# ------------------------------------------------------------

# 학습 중 epoch별 손실 값을 저장할 리스트
loss_history = []

# (위 학습 루프 안에서 total_loss를 loss_history.append(total_loss) 했다고 가정)

def plot_training_performance(loss_history):
    # 이 함수는 학습이 안정적으로 수렴했는지 확인하기 위한 그래프를 그린다.

    plt.figure(figsize=(8,4))
    # 그래프 크기 설정

    plt.plot(loss_history, label="Total Loss")
    # epoch에 따른 전체 손실 값

    plt.xlabel("Epoch")
    # x축은 학습 epoch

    plt.ylabel("Loss")
    # y축은 손실 값

    plt.title("Training Loss Curve (Hybrid PIN-ODE)")
    # 그래프 제목

    plt.legend()
    # 범례 표시

    plt.grid(True)
    # 격자 표시로 추세를 보기 쉽게 함

    plt.tight_layout()
    plt.show()
    # 그래프 출력

# ▶ 해석 포인트:
# - Loss가 초반에 빠르게 감소하고 이후 완만해지면 정상 학습
# - 진동이 크면 학습률 과다 또는 모델 불안정


In [30]:
# ------------------------------------------------------------
# 12. 예측 결과 + Physics Residual 시각화
# ------------------------------------------------------------

@torch.no_grad()
def run_inference(seq, title, fault_window=None):
    # 하나의 시계열(seq)에 대해
    # 1) 상태 예측
    # 2) Physics Residual 기반 이상 점수 계산
    # 3) 결과 시각화까지 한 번에 수행하는 함수

    X = torch.tensor(seq, dtype=torch.float32, device=device)
    # numpy → torch 변환

    U = torch.tensor(u[:, None], dtype=torch.float32, device=device)
    # 입력 u도 torch 변환

    # ---------- ODE Rollout ----------
    x_hat = X[0:1]
    preds = []

    for k in range(N):
        preds.append(x_hat)
        x_hat = rk4(model, x_hat, U[k:k+1])

    preds = torch.cat(preds).cpu().numpy()
    # 예측 시계열 (N,3)

    # ---------- Physics Residual ----------
    vel = X[:-1,1]
    acc_obs = (vel[1:] - vel[:-1]) / dt
    # 관측 가속도 근사

    _, fr_hat = model.dynamics(X[:-1], U[:-1])
    # 마찰 추정

    residual = (
        m * acc_obs.cpu().numpy()
        - (A * X[:-1,2].cpu().numpy()
           - c * vel.cpu().numpy()
           - fr_hat.squeeze().cpu().numpy())
    )
    # Physics Residual 계산

    anomaly_score = np.abs(residual)
    # 이상 점수는 residual 절댓값

    # ---------- 시각화 ----------
    plt.figure(figsize=(12,8))

    # (1) 위치 예측
    plt.subplot(3,1,1)
    plt.plot(t, seq[:,0], label="Observed x")
    plt.plot(t, preds[:,0], "--", label="Predicted x")
    plt.ylabel("Position")
    plt.legend()
    plt.title(title)

    # (2) 속도 예측
    plt.subplot(3,1,2)
    plt.plot(t, seq[:,1], label="Observed v")
    plt.plot(t, preds[:,1], "--", label="Predicted v")
    plt.ylabel("Velocity")
    plt.legend()

    # (3) Physics Residual (이상 점수)
    plt.subplot(3,1,3)
    plt.plot(t[:-1], anomaly_score, label="|Physics Residual|")

    if fault_window is not None:
        fs, fe = fault_window
        plt.axvspan(fs, fe, color="red", alpha=0.2, label="Fault Interval")

    plt.xlabel("Time (s)")
    plt.ylabel("Anomaly Score")
    plt.legend()

    plt.tight_layout()
    plt.show()

    return anomaly_score, fr_hat.squeeze().cpu().numpy()


In [31]:
# ------------------------------------------------------------
# 13. 정상 및 고장 시나리오 결과 비교
# ------------------------------------------------------------

# 정상 시나리오
an_normal, fr_normal = run_inference(
    train_trajs[0],
    title="Normal Operation"
)

# 마찰 고장 시나리오
an_friction, fr_friction = run_inference(
    test_fric,
    title="Friction Fault Scenario",
    fault_window=(12.0, 18.0)
)

# 압력 고장 시나리오
an_pressure, fr_pressure = run_inference(
    test_pres,
    title="Pressure Fault Scenario",
    fault_window=(12.0, 18.0)
)


ValueError: operands could not be broadcast together with shapes (1498,) (1499,) 

In [ ]:
# ------------------------------------------------------------
# 14. 결과 해석 자동 요약
# ------------------------------------------------------------

def interpret_result(name, anomaly, friction_hat, fault_window=None):
    # anomaly와 friction_hat을 기반으로
    # "무슨 고장인지"를 사람이 읽을 수 있는 문장으로 요약한다.

    print(f"\n[{name} 결과 해석]")

    if fault_window is not None:
        fs, fe = fault_window
        s = int(fs/dt)
        e = int(fe/dt)

        mean_anomaly = anomaly[s:e].mean()
        mean_friction = friction_hat[s:e].mean()
    else:
        mean_anomaly = anomaly.mean()
        mean_friction = friction_hat.mean()

    print(f"- 평균 Physics Residual: {mean_anomaly:.3f}")
    print(f"- 평균 추정 마찰 크기: {mean_friction:.3f}")

    # ---------- 해석 규칙 ----------
    if mean_anomaly < 1.0:
        print("▶ 설비 상태 판단: 정상")
        print("  - 물리 방정식 위반이 거의 없음")
        print("  - 예측과 관측이 물리적으로 일관됨")

    else:
        if mean_friction > np.percentile(fr_normal, 90):
            print("▶ 설비 상태 판단: 마찰 이상 가능성 높음")
            print("  - 압력은 충분하나 가속도가 부족")
            print("  - 가이드 마모, 윤활 불량, 기계적 저항 증가 의심")
        else:
            print("▶ 설비 상태 판단: 압력/유압 계통 이상 가능성")
            print("  - 마찰은 정상 수준")
            print("  - 밸브 응답 지연, 누설, 압력 형성 문제 의심")

# 결과 해석 출력
interpret_result("정상", an_normal, fr_normal)
interpret_result("마찰 고장", an_friction, fr_friction, fault_window=(12,18))
interpret_result("압력 고장", an_pressure, fr_pressure, fault_window=(12,18))


🧠 최종 결과 해석 요약 (보고서/회의용 문구)
✅ 정상

Physics Residual이 전 구간에서 낮음

모델 예측이 물리적으로 일관됨
→ 설비 정상 운전 상태

🔧 마찰 고장

Residual 급증 + friction̂ 항 증가

압력은 정상이나 운동이 둔화됨
→ 가이드 마모 / 윤활 불량 / 기계 저항 증가

🔩 압력 고장

Residual 증가 + 압력 예측 불일치

마찰은 정상
→ 밸브 지연 / 유압 누설 / 압력 형성 문제

### 정상(Normal):

- Physics Residual Anomaly Score가 전체적으로 낮고 안정적
- 예측(점선)이 관측(실선)을 큰 틀에서 잘 따라감

### 마찰 고장(Friction Fault):
- 고장 구간(12~18s)에서 Residual 급증
- Term decomposition에서 **|friction_hat|**가 상대적으로 커지는 경향
  → “마찰/저항 항 붕괴” 원인분류

### 압력 고장(Pressure Fault):
- 고장 구간에서 Residual 증가 + 예측 압력 궤적이 어긋남
- Jacobian(특히 d(dv)/dP) 패턴이 달라질 수 있음
  → “압력 동역학/구동력 전달 쪽 문제” 단서